<a href="https://colab.research.google.com/github/LizaHam123/sentiment-analysis-algerie-telecom/blob/main/spacynettoyage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas spacy tensorflow transformers
!python -m spacy download fr_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 56.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import re
import spacy
from tqdm import tqdm
import multiprocessing as mp
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import BertTokenizer
import numpy as np

# Configuration pour utiliser le GPU si disponible
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Utilisation de: {device}")

# Chargement du modèle SpaCy avec optimisations
nlp = spacy.load("fr_core_news_sm")

# OPTIMISATION 1: Désactiver les composants inutiles pour plus de vitesse
nlp.disable_pipes("parser", "ner")  # On garde seulement lemmatizer et POS tagger

# OPTIMISATION 2: Configurer pour traitement par batch
nlp.max_length = 1500000  # Augmenter la limite pour gros textes

print(" SpaCy optimisé chargé")

print("Montage de Google Drive...")
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive monté")

# 1. Chargement des données
df = pd.read_csv("resultat_analyse.csv", encoding='utf-8')
print(f" Données chargées: {len(df)} commentaires")

# 2. Nettoyage de base commun (RAPIDE)
def clean_common(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]+>|http\S+|www\.\S+|@\w+|#\w+', '', text)
    text = re.sub(r'[^\w\s!?.,]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print(" Nettoyage de base...")
df['text_clean'] = df['commentaire'].apply(clean_common)
print(" Nettoyage de base terminé")

# 3. TRAITEMENT SPACY ULTRA-OPTIMISÉ
def process_batch_spacy(texts, batch_size=1000):
    """Traitement par batch pour maximiser la vitesse"""
    all_results = []

    # Traitement par batch avec barre de progression
    for i in tqdm(range(0, len(texts), batch_size), desc=" Traitement SpaCy"):
        batch = texts[i:i+batch_size]

        # Pré-traitement du batch
        processed_batch = []
        for text in batch:
            if not isinstance(text, str) or text.strip() == "":
                processed_batch.append("")
                continue

            # Corrections rapides
            text = text.lower()
            text = re.sub(r'(\w)\1{2,}', r'\1', text)  # Répétitions
            processed_batch.append(text)

        # Traitement SpaCy par batch
        docs = list(nlp.pipe(processed_batch, batch_size=50, n_process=1))

        # Extraction des lemmes
        batch_results = []
        for doc in docs:
            if not doc.text.strip():
                batch_results.append("")
                continue

            cleaned_words = []
            for token in doc:
                if (not token.is_punct and
                    not token.is_space and
                    not token.is_stop and
                    len(token.text) > 2 and
                    token.text.isalpha()):
                    cleaned_words.append(token.lemma_)

            batch_results.append(' '.join(cleaned_words))

        all_results.extend(batch_results)

    return all_results

# Application du traitement optimisé
print(" Traitement SpaCy par batch (ultra-rapide)...")
df['text_lstm'] = process_batch_spacy(df['text_clean'].tolist(), batch_size=500)
print(" Traitement SpaCy terminé")

# 4. TOKENISATION OPTIMISÉE
print(" Tokenisation LSTM...")
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(df['text_lstm'])
sequences = tokenizer.texts_to_sequences(df['text_lstm'])

# Padding optimisé
print(" Padding des séquences...")
padded_sequences = pad_sequences(sequences, maxlen=100, padding='post', truncating='post')
df['sequences_lstm'] = list(padded_sequences)
print(" Tokenisation LSTM terminée")

# 5. TRAITEMENT BERT OPTIMISÉ
def process_bert_batch(texts, batch_size=1000):
    """Traitement BERT par batch"""
    results = []

    for i in tqdm(range(0, len(texts), batch_size), desc=" Traitement BERT"):
        batch = texts[i:i+batch_size]
        batch_processed = [re.sub(r'[^\w\s!?.]', ' ', str(text)) for text in batch]
        results.extend(batch_processed)

    return results

print(" Traitement BERT par batch...")
df['text_bert'] = process_bert_batch(df['commentaire'].tolist())

# Tokenisation BERT optimisée
print(" Tokenisation BERT...")
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-uncased')

def tokenize_bert_batch(texts, batch_size=500):
    """Tokenisation BERT par batch"""
    results = []

    for i in tqdm(range(0, len(texts), batch_size), desc=" Tokenisation BERT"):
        batch = texts[i:i+batch_size]

        batch_tokens = []
        for text in batch:
            tokens = bert_tokenizer.encode_plus(
                text,
                max_length=128,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            batch_tokens.append(tokens)

        results.extend(batch_tokens)

    return results

df['bert_input'] = tokenize_bert_batch(df['text_bert'].tolist())
print(" Tokenisation BERT terminée")

# 6. FORMATAGE FASTTEXT OPTIMISÉ
print(" Génération FastText...")
def format_fasttext(text, label):
    text = re.sub(r'\s+', ' ', str(text)).strip()
    return f"__label__{label} {text}\n"

with open("fasttext_data.txt", "w", encoding="utf-8") as f:
    for text, label in tqdm(zip(df['text_clean'], df['sentiment']),
                           total=len(df), desc="FastText"):
        f.write(format_fasttext(text.lower(), label))

# 7. SAUVEGARDE OPTIMISÉE
print("Sauvegarde...")
output_df = df[['text_lstm', 'sequences_lstm', 'text_bert', 'bert_input', 'sentiment', 'probleme']]

# Sauvegarde par chunks pour éviter les problèmes de mémoire
chunk_size = 1000
for i in range(0, len(output_df), chunk_size):
    chunk = output_df.iloc[i:i+chunk_size]
    mode = 'w' if i == 0 else 'a'
    header = True if i == 0 else False
    chunk.to_csv("processed_data.csv", mode=mode, header=header, index=False)

print(" TRAITEMENT ULTRA-RAPIDE TERMINÉ!")
print(f" {len(df)} commentaires traités")

# SAUVEGARDE GOOGLE DRIVE
print("Sauvegarde sur Google Drive...")
import shutil

try:
    # Copier vers Google Drive
    shutil.copy('processed_data.csv', '/content/drive/MyDrive/processed_data.csv')
    shutil.copy('fasttext_data.txt', '/content/drive/MyDrive/fasttext_data.txt')

    print("Fichiers sauvegardés sur Google Drive:")
    print("  • /content/drive/MyDrive/processed_data.csv")
    print("  • /content/drive/MyDrive/fasttext_data.txt")

except Exception as e:
    print(f"Erreur sauvegarde Drive: {e}")

print(" Fichiers générés:")
print("  • processed_data.csv (local + Drive)")
print("  • fasttext_data.txt (local + Drive)")

# 8. STATISTIQUES DE PERFORMANCE
print(f"\n STATISTIQUES:")
print(f"• Commentaires traités: {len(df):,}")
print(f"• Longueur moyenne avant: {df['text_clean'].str.len().mean():.1f} caractères")
print(f"• Longueur moyenne après: {df['text_lstm'].str.len().mean():.1f} caractères")
print(f"• Mots moyens par commentaire: {df['text_lstm'].str.split().str.len().mean():.1f}")

# Aperçu rapide
print(f"\n APERÇU (3 exemples):")
for i in range(min(3, len(df))):
    print(f"\n{i+1}. Original: {df['commentaire'].iloc[i][:80]}...")
    print(f"   Nettoyé:  {df['text_lstm'].iloc[i][:80]}...")

 Utilisation de: cpu
 SpaCy optimisé chargé
Montage de Google Drive...
Mounted at /content/drive
Google Drive monté
 Données chargées: 9237 commentaires
 Nettoyage de base...
 Nettoyage de base terminé
 Traitement SpaCy par batch (ultra-rapide)...


 Traitement SpaCy: 100%|██████████| 19/19 [00:22<00:00,  1.19s/it]


 Traitement SpaCy terminé
 Tokenisation LSTM...
 Padding des séquences...
 Tokenisation LSTM terminée
 Traitement BERT par batch...


 Traitement BERT: 100%|██████████| 10/10 [00:00<00:00, 103.91it/s]

 Tokenisation BERT...


...

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

 Tokenisation BERT: 100%|██████████| 19/19 [00:14<00:00,  1.31it/s]


 Tokenisation BERT terminée
 Génération FastText...


FastText: 100%|██████████| 9237/9237 [00:00<00:00, 21435.37it/s]


Sauvegarde...
 TRAITEMENT ULTRA-RAPIDE TERMINÉ!
 9237 commentaires traités
Sauvegarde sur Google Drive...
Fichiers sauvegardés sur Google Drive:
  • /content/drive/MyDrive/processed_data.csv
  • /content/drive/MyDrive/fasttext_data.txt
 Fichiers générés:
  • processed_data.csv (local + Drive)
  • fasttext_data.txt (local + Drive)

 STATISTIQUES:
• Commentaires traités: 9,237
• Longueur moyenne avant: 87.7 caractères
• Longueur moyenne après: 45.4 caractères
• Mots moyens par commentaire: 6.2

 APERÇU (3 exemples):

1. Original: Plantez et vous récolterez.,...
   Nettoyé:  planter récolter...

2. Original: Bravo et merci à tous les employés d’Algérie Télécom.,...
   Nettoyé:  bravo employé algérie télécom...

3. Original: Votre entreprise ne pense plus qu’aux abonnés Fibre !!!,...
   Nettoyé:  entreprise abonné fibre...

 FINI ! Gain de vitesse: ~5-10x plus rapide !
